## Most Popular Client by Call Event Engagement

Select the most popular client_id based on a count of the number of users who have at least 50% of their events from the following list: 'video call received', 'video call sent', 'voice call received', 'voice call sent'.

🌀By solving this, you'll learn how to use Mutiple Cte, Group by, Join. Give it a try and share the output! 👇

In [0]:
%skip
%sql
CREATE TABLE ska_catalog2.bronze.fact_event (id BIGINT PRIMARY KEY, time_id TIMESTAMP, user_id VARCHAR(50), customer_id VARCHAR(50), client_id VARCHAR(50), event_type VARCHAR(50), event_id BIGINT);

INSERT INTO  ska_catalog2.bronze.fact_event (id, time_id, user_id, customer_id, client_id, event_type, event_id) VALUES (1, '2024-02-01 10:00:00', 'U1', 'C1', 'CL1', 'video call received', 101), (2, '2024-02-01 10:05:00', 'U1', 'C1', 'CL1', 'video call sent', 102), (3, '2024-02-01 10:10:00', 'U1', 'C1', 'CL1', 'message sent', 103), (4, '2024-02-01 11:00:00', 'U2', 'C2', 'CL2', 'voice call received', 104), (5, '2024-02-01 11:10:00', 'U2', 'C2', 'CL2', 'voice call sent', 105), (6, '2024-02-01 11:20:00', 'U2', 'C2', 'CL2', 'message received', 106), (7, '2024-02-01 12:00:00', 'U3', 'C3', 'CL1', 'video call sent', 107), (8, '2024-02-01 12:15:00', 'U3', 'C3', 'CL1', 'voice call received', 108), (9, '2024-02-01 12:30:00', 'U3', 'C3', 'CL1', 'voice call sent', 109), (10, '2024-02-01 12:45:00', 'U3', 'C3', 'CL1', 'video call received', 110);


In [0]:
SELECT * FROM  ska_catalog2.bronze.fact_event

In [0]:
WITH user_event_counts AS (
  SELECT
    user_id,
    COUNT(*) AS total_events
  FROM ska_catalog2.bronze.fact_event
  GROUP BY user_id
),
filtered_event_counts AS (
  SELECT
    user_id,
    client_id,
    COUNT(*) AS matched_events
  FROM  ska_catalog2.bronze.fact_event
  WHERE event_type IN ('video call received', 'video call sent','voice call received','voice call sent')
  GROUP BY user_id, client_id
),
qualified_users AS (
  SELECT f.user_id,
    f.client_id
  FROM filtered_event_counts f
  JOIN user_event_counts u ON f.user_id = u.user_id
  WHERE f.matched_events >= 0.5 * u.total_events
),
client_popularity AS (
  SELECT
    client_id,
    COUNT(DISTINCT user_id) AS user_count
  FROM qualified_users
  GROUP BY client_id
)
SELECT client_id
FROM client_popularity
ORDER BY user_count DESC
LIMIT 1;